In [1]:
!pip install transformers datasets scikit-learn -q

In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GPU available: True
Device: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd gdrive/My\ Drive/Colab\ Notebooks/emotion_project

Mounted at /content/gdrive
/content/gdrive/My Drive/Colab Notebooks/emotion_project


In [11]:
from datasets import load_from_disk
from torch.utils.data import Dataset, DataLoader

tokenized = load_from_disk("processed/go_emotions_tokenized")
# Only pre-format fixed-length tensor columns; 'labels' (variable-length list) is
# converted to a multi-hot vector inside EmotionDataset.__getitem__
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])

label_names = tokenized['train'].features['labels'].feature.names
NUM_LABELS  = len(label_names)
print(tokenized)
print(f"{NUM_LABELS} classes")

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5427
    })
})
28 classes


In [6]:
class EmotionDataset(Dataset):
    def __init__(self, hf_split, num_labels=28):
        self.data       = hf_split
        self.num_labels = num_labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        # Convert variable-length label list to a fixed-size multi-hot float vector
        label_vec = torch.zeros(self.num_labels, dtype=torch.float)
        for lbl in row['labels']:
            label_vec[lbl] = 1.0
        return {
            'input_ids':      row['input_ids'],
            'attention_mask': row['attention_mask'],
            'token_type_ids': row['token_type_ids'],
            'labels':         label_vec,
        }

train_loader = DataLoader(EmotionDataset(tokenized['train'],      NUM_LABELS), batch_size=32, shuffle=True)
val_loader   = DataLoader(EmotionDataset(tokenized['validation'], NUM_LABELS), batch_size=64, shuffle=False)
test_loader  = DataLoader(EmotionDataset(tokenized['test'],       NUM_LABELS), batch_size=64, shuffle=False)

### Fine-Tuning BERT
Load `bert-base-uncased` with a fresh 28-class classification head and fine-tune on the training set.

In [7]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = 1 / (1 + np.exp(-logits))        # sigmoid
    preds  = (probs >= 0.5).astype(int)
    labels = labels.astype(int)

    micro_f1    = f1_score(labels, preds, average='micro', zero_division=0)
    macro_f1    = f1_score(labels, preds, average='macro', zero_division=0)
    exact_match = float((preds == labels).all(axis=1).mean())
    return {
        'micro_f1':    micro_f1,
        'macro_f1':    macro_f1,
        'exact_match': exact_match,
    }

In [8]:
from transformers import BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

model = BertForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
).to(DEVICE)

EPOCHS    = 10
LR        = 3e-5
WARMUP    = 0.1
PATIENCE  = 3

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps)

print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total steps: 13570 | Warmup steps: 1357


#### Sanity Check — 200 samples
Run one forward + backward pass on a small subset to confirm no shape errors before full training.

In [12]:
from torch.utils.data import Subset

small_loader = DataLoader(Subset(EmotionDataset(tokenized['train'], NUM_LABELS), range(200)),
                          batch_size=32, shuffle=False)

model.train()
for batch in small_loader:
    input_ids      = batch['input_ids'].to(DEVICE)
    attention_mask = batch['attention_mask'].to(DEVICE)
    token_type_ids = batch['token_type_ids'].to(DEVICE)
    labels         = batch['labels'].to(DEVICE)  # float multi-hot [batch, 28]

    outputs = model(input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    labels=labels)
    outputs.loss.backward()
    optimizer.zero_grad()

print(f"Sanity check passed — loss: {outputs.loss.item():.4f} | logits shape: {outputs.logits.shape}")

Sanity check passed — loss: 0.6918 | logits shape: torch.Size([8, 28])


#### Full Training Loop

In [ ]:
import os
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, default_data_collator

SAVE_DIR = "checkpoints"

# Re-initialise model cleanly before full training
model = BertForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
).to(DEVICE)

# Freeze the bottom N transformer layers + embeddings, fine-tuning only the top layers and the classification head
N_FREEZE = 6  # freeze bottom N of 12 transformer layers
for name, param in model.named_parameters():
    if 'embeddings' in name:
        param.requires_grad = False
    elif 'encoder.layer.' in name:
        layer_idx = int(name.split('encoder.layer.')[1].split('.')[0])
        if layer_idx < N_FREEZE:
            param.requires_grad = False

frozen = sum(1 for p in model.parameters() if not p.requires_grad)
total  = sum(1 for p in model.parameters())
print(f"Frozen {frozen}/{total} parameter groups")

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=WARMUP,
    max_grad_norm=1.0,
    eval_strategy="epoch",       # evaluate at the end of every epoch
    save_strategy="epoch",       # save a checkpoint at the end of every epoch
    save_total_limit=3,          # keep only the 3 most recent checkpoints + best
    load_best_model_at_end=True, # restore best checkpoint when training ends
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=EmotionDataset(tokenized['train'],      NUM_LABELS),
    eval_dataset =EmotionDataset(tokenized['validation'], NUM_LABELS),
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

# To resume from the latest saved checkpoint after an interruption, replace
# trainer.train() with:  trainer.train(resume_from_checkpoint=True)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecate

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Exact Match
1,0.123273,0.113391,0.435386,0.126327,0.314596
2,0.088987,0.087856,0.539047,0.327149,0.410984
3,0.075913,0.084710,0.573963,0.395401,0.456137
4,0.065149,0.088428,0.570786,0.437713,0.459270
5,0.051243,0.094336,0.573207,0.454731,0.467748
6,0.042997,0.103018,0.571211,0.441005,0.468301
7,0.035268,0.108425,0.562610,0.454954,0.455584
8,0.028493,0.115151,0.562541,0.469861,0.451898
9,0.024753,0.118769,0.562307,0.472592,0.449871
10,0.021254,0.119841,0.561966,0.471766,0.449871


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=13570, training_loss=0.06699159704944889, metrics={'train_runtime': 3244.5214, 'train_samples_per_second': 133.795, 'train_steps_per_second': 4.182, 'total_flos': 2.85607930586112e+16, 'train_loss': 0.06699159704944889, 'epoch': 10.0})

In [2]:
import pandas as pd

# Trainer logs one dict per eval event — filter to rows that have eval metrics
eval_logs = [l for l in trainer.state.log_history if 'eval_macro_f1' in l]
df = pd.DataFrame(eval_logs)[['epoch', 'eval_loss', 'eval_macro_f1', 'eval_micro_f1', 'eval_exact_match']]
df.columns = ['epoch', 'loss', 'macro_f1', 'micro_f1', 'exact_match']
print(df.to_string(index=False))

NameError: name 'trainer' is not defined

In [1]:
import os, json

os.makedirs("results", exist_ok=True)

RUN_NAME = "initial"   # change to e.g. "freeze6" before saving a fine-tuned run

# --- per-epoch history ---
df['run'] = RUN_NAME
df.to_csv(f"results/run_{RUN_NAME}.csv", index=False)

# --- best-epoch summary (one row per run, for the Week 5 comparison table) ---
best_row = df.loc[df['macro_f1'].idxmax()]
summary  = {
    'run':         RUN_NAME,
    'best_epoch':  int(best_row['epoch']),
    'loss':        round(float(best_row['loss']),        6),
    'macro_f1':    round(float(best_row['macro_f1']),    6),
    'micro_f1':    round(float(best_row['micro_f1']),    6),
    'exact_match': round(float(best_row['exact_match']), 6),
}
with open(f"results/best_{RUN_NAME}.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved  results/run_{RUN_NAME}.csv")
print(f"Saved  results/best_{RUN_NAME}.json")
print()
for k, v in summary.items():
    print(f"  {k}: {v}")

NameError: name 'df' is not defined